In [1]:
import torch
import gc

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  —  {p.total_memory / 1024**3:.1f} GB VRAM")
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU disponibili: {torch.cuda.device_count()}")

  [0] Tesla T4  —  14.6 GB VRAM
GPU disponibili: 2
  [1] Tesla T4  —  14.6 GB VRAM
GPU disponibili: 2


In [2]:
%%bash
pip install -q basicsr facexlib lpips einops timm scikit-image pyyaml tensorboard pyiqa

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 11.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 112.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9

In [3]:
import os, sys, shutil, yaml, glob, subprocess

# ─── MODIFICA QUESTI PATH ────────────────────────────────────────────────────
CODE_SRC  = "/kaggle/input/datasets/francescoardolino02/fcadreamuhdtrain/FcaDreamuhdTrain"
DATA_ROOT = "/kaggle/input/datasets/francescoardolino02/uhd-ll-crop-1024/uhd_ll_3k_crop"   # contiene training_set/ e testing_set/
# ─────────────────────────────────────────────────────────────────────────────

CODE_DEST = "/kaggle/working/FcaDreamuhd"
EXP_DIR   = "/kaggle/working/experiments"   # ← qui vengono salvati i checkpoint

if os.path.exists(CODE_DEST):
    shutil.rmtree(CODE_DEST)
shutil.copytree(CODE_SRC, CODE_DEST)
os.makedirs(EXP_DIR, exist_ok=True)
print(f"✅ Codice copiato in: {CODE_DEST}")
print(f"✅ Esperimenti in   : {EXP_DIR}")

✅ Codice copiato in: /kaggle/working/FcaDreamuhd
✅ Esperimenti in   : /kaggle/working/experiments


In [4]:
VAE_TRAINED = "/kaggle/input/datasets/francescoardolino02/experiments/experiments/VAE_LL/models/net_g_45000.pth"

import os
if not os.path.exists(VAE_TRAINED):
    raise FileNotFoundError(f"Errore: non trovo il VAE in {VAE_TRAINED}")
print(f"✅ VAE trainato trovato: {VAE_TRAINED}")

✅ VAE trainato trovato: /kaggle/input/datasets/francescoardolino02/experiments/experiments/VAE_LL/models/net_g_45000.pth


In [5]:
DREAM_YML_SRC  = os.path.join(CODE_DEST, "options/DreamUHD_LL.yml")

In [6]:
DREAM_YML_DEST = "/kaggle/working/DreamUHD_LL_kaggle.yml"

with open(DREAM_YML_SRC, "r") as f:
    dream_cfg = yaml.safe_load(f)

dream_cfg["name"] = "FcaDreamUHD_Stage2"

DREAM_NAME    = dream_cfg.get("name", "FcaDreamUHD")
DREAM_EXP_DIR = os.path.join(EXP_DIR, DREAM_NAME)
os.makedirs(DREAM_EXP_DIR, exist_ok=True)

# Dataset
dream_cfg["datasets"]["train"]["dataroot_gt"] = os.path.join(DATA_ROOT, "training_set/gt")
dream_cfg["datasets"]["train"]["dataroot_lq"] = os.path.join(DATA_ROOT, "training_set/input")
dream_cfg["datasets"]["val"]["dataroot_gt"]   = os.path.join(DATA_ROOT, "testing_set/gt")
dream_cfg["datasets"]["val"]["dataroot_lq"]   = os.path.join(DATA_ROOT, "testing_set/input")

# VAE prodotto nello Stage 1
dream_cfg["network_g"]["vae_weight"] = VAE_TRAINED
dream_cfg["network_g"]["config"]     = os.path.join(CODE_DEST, "options/VAE_LL.yml")

# ⚠️ Forza i path esperimenti in /kaggle/working/
dream_cfg["path"]["experiments_root"] = DREAM_EXP_DIR
dream_cfg["path"]["models"]           = os.path.join(DREAM_EXP_DIR, "models")
dream_cfg["path"]["training_states"]  = os.path.join(DREAM_EXP_DIR, "training_states")
dream_cfg["path"]["log"]              = DREAM_EXP_DIR
dream_cfg["path"]["visualization"]    = os.path.join(DREAM_EXP_DIR, "visualization")

# Nessun pretrain — da zero come gli autori
dream_cfg["path"]["pretrain_network_g"] = None
dream_cfg["path"]["pretrain_network_d"] = None
dream_cfg["path"]["resume_state"]       = None
# Per resume Stage 2:
# dream_cfg["path"]["resume_state"] = f"{DREAM_EXP_DIR}/training_states/25000.state"

with open(DREAM_YML_DEST, "w") as f:
    yaml.dump(dream_cfg, f, default_flow_style=False, allow_unicode=True)

print(f"✅ DreamUHD config: {DREAM_YML_DEST}")
print(f"   experiments_root : {dream_cfg['path']['experiments_root']}")
print(f"   vae_weight       : {dream_cfg['network_g']['vae_weight']}")
print(f"   resume_state     : {dream_cfg['path']['resume_state']}")

✅ DreamUHD config: /kaggle/working/DreamUHD_LL_kaggle.yml
   experiments_root : /kaggle/working/experiments/FcaDreamUHD_Stage2
   vae_weight       : /kaggle/input/datasets/francescoardolino02/experiments/experiments/VAE_LL/models/net_g_45000.pth
   resume_state     : None


In [7]:
TRAIN_PY = "/kaggle/working/FcaDreamuhd/basicsr/train.py"

In [8]:
import os

dream_arch_path = "/kaggle/working/FcaDreamuhd/basicsr/archs/DreamUHD_arch.py"

with open(dream_arch_path, 'r') as f:
    codice = f.read()

codice = codice.replace('["vanilla","FE-block","FE-block2"]', '["vanilla","FE-block","FE-block2","FEblock"]')
codice = codice.replace('res_type == "FE-block":', 'res_type in ["FE-block", "FEblock"]:')

with open(dream_arch_path, 'w') as f:
    f.write(codice)

print("✅ Fix applicato a DreamUHD_arch.py! Ora puoi lanciare lo Stage 2.")

✅ Fix applicato a DreamUHD_arch.py! Ora puoi lanciare lo Stage 2.


In [9]:
cmd = [
    "torchrun",
    "--nproc_per_node=2",
    "--master_port=29500",
    TRAIN_PY,
    "-opt", DREAM_YML_DEST,
    "--launcher", "pytorch",
]
print("[STAGE 2] Comando:", " ".join(cmd))
print("─" * 70)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
    env={**os.environ, "PYTHONPATH": CODE_DEST},
)
for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()
print(f"\nExit code: {proc.returncode}")

[STAGE 2] Comando: torchrun --nproc_per_node=2 --master_port=29500 /kaggle/working/FcaDreamuhd/basicsr/train.py -opt /kaggle/working/DreamUHD_LL_kaggle.yml --launcher pytorch
──────────────────────────────────────────────────────────────────────


W0520 18:25:44.272000 73 torch/distributed/run.py:852] 


W0520 18:25:44.272000 73 torch/distributed/run.py:852] *****************************************


W0520 18:25:44.272000 73 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 


W0520 18:25:44.272000 73 torch/distributed/run.py:852] *****************************************


...

Test 425_UHD_LL_center: 100%|██████████| 150/150 [01:52<00:00,  1.34image/s] 


Test 425_UHD_LL_center: 100%|██████████| 150/150 [01:52<00:00,  1.33image/s]


2026-05-21 06:17:02,928 INFO: [FcaDr..][epoch: 18, iter:  18,100, lr:(7.650e-04,)] [eta: 1 day, 4:01:55, time (data): 2.269 (0.008)] l_pix: 5.0952e-02 l_freq: 1.0446e+00 


2026-05-21 06:20:51,071 INFO: [FcaDr..][epoch: 18, iter:  18,200, lr:(7.646e-04,)] [eta: 1 day, 3:57:43, time (data): 2.280 (0.009)] l_pix: 5.2438e-02 l_freq: 1.4499e+00 


2026-05-21 06:24:39,479 INFO: [FcaDr..][epoch: 18, iter:  18,300, lr:(7.641e-04,)] [eta: 1 day, 3:53:32, time (data): 2.293 (0.008)] l_pix: 3.6484e-02 l_freq: 8.6795e-01 
